# 02 · Generate and capture: COS vs DICT vs CONTROL

**Requires Apple Silicon + MLX.** Regenerates the arrays shipped in `data/`.

For each prompt in `prompts/` the model greedily generates up to 100 tokens. At **every step** and
**every layer** we record:

* `attn_window` `(steps, layers, 100, 100)`: head 17's attention map, 3×3 average-pooled, bottom-right
  100×100 block (the most recent 100 query positions attending to the most recent 100 keys).
* `last_hidden` `(steps, layers, 3072)`: the last token's residual stream after the model's final RMSNorm.

Three prompts: **COS** (symbol-encoded yearly trend), **DICT** (same data as a raw dict), and
**CONTROL** (unrelated narrative prose).

Each step re-runs the full sequence (no KV cache), so expect several minutes per prompt.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
from mlx_lm import load

from attnscope.scope import generate_and_capture, load_prompt

MODEL_ID = "mlx-community/Llama-3.2-3B-Instruct"
HEAD = 17          # 0-indexed
WINDOW = 100
MAX_NEW_TOKENS = 100
OUT_DIR = ROOT / "data"
OUT_DIR.mkdir(parents=True, exist_ok=True)

model, tokenizer = load(MODEL_ID)

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

In [2]:
results = {}
for name in ("cos", "dict", "control"):
    prompt = load_prompt(ROOT / "prompts" / f"{name}.txt")
    res = generate_and_capture(model, tokenizer, prompt, max_new_tokens=MAX_NEW_TOKENS, head=HEAD, window=WINDOW)
    res.save(OUT_DIR / f"{name}.npz")
    results[name] = res
    print(f"=== {name.upper()} ===  attn {res.attn_window.shape}  hidden {res.last_hidden.shape}")
    print(res.text[len(prompt):])
    print()

  0%|          | 0/100 [00:00<?, ?it/s]

=== COS ===  attn (100, 28, 100, 100)  hidden (100, 28, 3072)
  The airline's passenger data shows a steady increase in the number of passengers over the years, with a notable
  acceleration in the mid-1950s. The data suggests that the airline experienced a significant growth spurt in the
  late 1950s and early 1960s, with a peak in 1959 and 1960. This growth was likely driven by the expansion of
  air travel and the increasing popularity of air travel among the general public. The data



  0%|          | 0/100 [00:00<?, ?it/s]

=== DICT ===  attn (100, 28, 100, 100)  hidden (100, 28, 3072)
 The data shows a steady increase in the number of passengers from 1949 to 1959. The mean number of passengers per year increases from 126.67 in 1949 to 476.17 in 1960. The sum of passengers also increases steadily, with the highest number of passengers in 1959. The mean number of passengers per year increases by 49.5% from 1949 to 1959. The sum of passengers increases by 186.5%



  0%|          | 0/100 [00:00<?, ?it/s]

=== CONTROL ===  attn (100, 28, 100, 100)  hidden (100, 28, 3072)
.
The flight was delayed, and he was stuck in the airport for hours. The kids were getting restless, and his wife was getting frustrated. The stress was building up, and he was starting to feel like he was going to lose his mind. He tried to stay calm, but it was getting harder and harder. He thought to himself, “Why did I have to get stuck in this airport? Why did I have to be the one to deal with this? Why did I have to be

